# FGI Video-only tr?n dataset crop m?i

Notebook n?y d?ng **c?ng dataset, c?ng split v? c?ng crop m?i** v?i FGI audio?visual hi?n t?i, nh?ng lo?i b? ho?n to?n audio.

Model gi? backbone 3D ResNet h?nh ?nh c?a FGI v? thay ph?n fusion audio?visual b?ng m?t classifier video-only. K?t qu? c? th? so s?nh tr?c ti?p v?i FGI audio?visual ?? ?o ??ng g?p c?a audio.

In [ ]:
# Colab: Runtime -> Change runtime type -> GPU
!nvidia-smi -L
!pip -q install scikit-learn pandas opencv-python-headless tqdm

In [ ]:
# GI? NGUY?N CELL T?I/GI?I N?N DATASET C?A B?N
import os, sys, glob, tarfile, shutil, random, json, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FILE_ID = '1f92-OG4-TlEIKBIYJXUkFLis_jE3rOR_'
DRIVE_FILE = '/content/drive/MyDrive/pbl6_mouth/dfdc_processed.tar'
DATA_DIR = '/content/lipforensics_data'  # Legacy fallback
WORK_DIR = '/mnt/local-scratch/work'
REPO_DIR = '/content/LipForensics'
OUT_DIR = '/content/drive/MyDrive/fgi_video_only_runs'

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_FILE), exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

if os.path.exists(DRIVE_FILE) and os.path.getsize(DRIVE_FILE) < 100000000:
    print('T?p tar qu? nh?, x?a ?? t?i l?i.')
    os.remove(DRIVE_FILE)

if not os.path.exists(DRIVE_FILE):
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import io
    drive_service = build('drive', 'v3')
    request = drive_service.files().get_media(fileId=DRIVE_FILE_ID)
    fh = io.FileIO(DRIVE_FILE, 'wb')
    downloader = MediaIoBaseDownload(fh, request, chunksize=1024*1024*10)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status: print(f'T?i: {int(status.progress()*100)}%')
    fh.close()

if not (os.path.exists(os.path.join(WORK_DIR, 'train_split.csv')) and os.path.isdir(os.path.join(WORK_DIR, 'train', 'pytmp'))):
    tar_size = os.path.getsize(DRIVE_FILE)
    free = shutil.disk_usage('/mnt/local-scratch').free
    print(f'Tar: {tar_size/1024**3:.2f} GB | Free: {free/1024**3:.2f} GB')
    assert tar_size > 100000000, 'T?i tar th?t b?i ho?c file ch?a ???c chia s? ??ng quy?n.'
    r = subprocess.run(['tar', '-xf', DRIVE_FILE, '-C', WORK_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
    assert r.returncode == 0, 'Gi?i n?n tar th?t b?i.'

print('WORK_DIR:', WORK_DIR)
print('N?i dung:', os.listdir(WORK_DIR)[:10])

In [ ]:
# Ki?m tra v? chu?n h?a split; kh?ng t?o l?i n?u tar ?? c? split c?a FGI
import csv, glob, shutil, random
CSV_NAMES = ['train_split.csv','valid_split.csv','test_imbalance_split.csv']
pytmp_target = os.path.join(WORK_DIR,'train','pytmp')
if not os.path.isdir(pytmp_target):
    hits = glob.glob(os.path.join(WORK_DIR,'**','pytmp'),recursive=True)
    if hits:
        os.makedirs(os.path.dirname(pytmp_target),exist_ok=True); shutil.move(hits[0],pytmp_target)
assert os.path.isdir(pytmp_target), 'Kh?ng t?m th?y train/pytmp.'
for name in CSV_NAMES:
    dst=os.path.join(WORK_DIR,name)
    if not os.path.exists(dst):
        hits=glob.glob(os.path.join(WORK_DIR,'**',name),recursive=True)
        if hits: shutil.move(hits[0],dst)
    assert os.path.exists(dst), f'Thi?u {name}.'
    rows=[]
    with open(dst,newline='') as f:
        rows=list(csv.reader(f))
    fixed=[]
    for row in rows:
        if len(row)<3: continue
        p=row[0].replace('\\','/')
        k=p.find('train/pytmp/')
        if k>=0: p=p[k:]
        fixed.append([p,row[1],row[2]])
    with open(dst,'w',newline='') as f: csv.writer(f).writerows(fixed)
    print(name,len(fixed),'chunks')
print('Split gi? nguy?n t? FGI; audio column ch? ???c b? qua.')

In [ ]:
# FGI visual backbone + video-only classifier
import os, csv, random, math, json, cv2, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, log_loss
from tqdm.auto import tqdm

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_DIM=224; FRAMES=30; BATCH_SIZE=4; NUM_WORKERS=2

def conv3x3x3(a,b,stride=1): return nn.Conv3d(a,b,3,stride=stride,padding=1,bias=False)
class BasicBlock3d(nn.Module):
    def __init__(self,a,b,stride=1,downsample=None,final_relu=True):
        super().__init__(); self.c1=conv3x3x3(a,b,stride); self.n1=nn.BatchNorm3d(b); self.c2=conv3x3x3(b,b); self.n2=nn.BatchNorm3d(b); self.down=downsample; self.final_relu=final_relu
    def forward(self,x):
        r=x; x=F.relu(self.n1(self.c1(x)),inplace=True); x=self.n2(self.c2(x));
        if self.down is not None: r=self.down(r)
        x=x+r; return F.relu(x,inplace=True) if self.final_relu else x
class FGIVisualBackbone(nn.Module):
    def __init__(self):
        super().__init__(); self.inplanes=64
        self.conv=nn.Conv3d(3,64,(1,7,7),stride=(1,2,2),padding=(0,3,3),bias=False); self.bn=nn.BatchNorm3d(64)
        self.pool=nn.MaxPool3d((1,3,3),stride=(1,2,2),padding=(0,1,1))
        self.l1=self.layer(64,2); self.l2=self.layer(128,2,stride=2,final_block=True)
    def layer(self,planes,blocks,stride=1,final_block=False):
        down=None
        if stride!=1 or self.inplanes!=planes: down=nn.Sequential(nn.Conv3d(self.inplanes,planes,1,stride=stride,bias=False),nn.BatchNorm3d(planes))
        layers=[BasicBlock3d(self.inplanes,planes,stride,down,final_relu=True)]; self.inplanes=planes
        for i in range(1,blocks): layers.append(BasicBlock3d(planes,planes,final_relu=(i<blocks-1)))
        return nn.Sequential(*layers)
    def forward(self,x): return self.l2(self.l1(self.pool(F.relu(self.bn(self.conv(x)),inplace=True))))
class FGIVideoOnly(nn.Module):
    def __init__(self):
        super().__init__(); self.backbone=FGIVisualBackbone(); self.classifier=nn.Sequential(nn.LayerNorm(128),nn.Dropout(.3),nn.Linear(128,1))
    def forward(self,x):
        # x: [B, 1, 3, 30, 224, 224], same video layout as FGI
        x=x[:,0]; feat=self.backbone(x); feat=feat.mean(dim=(2,3,4)); return self.classifier(feat).squeeze(1)

print('Device:',DEVICE,'parameters:',sum(p.numel() for p in FGIVideoOnly().parameters()))

In [ ]:
class FGIVideoDataset(Dataset):
    def __init__(self,csv_path,train=False):
        self.train=train; self.items=[]
        with open(csv_path,newline='') as f:
            for row in csv.reader(f):
                if len(row)>=3:
                    label=int(row[2].lower() in ('fake','1','true')); rel=row[0].replace('\\','/')
                    if rel.startswith('/'): folder=Path(rel)
                    else: folder=Path(WORK_DIR)/rel
                    self.items.append((folder,label,folder.parts[-2]))
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        folder,label,video_id=self.items[i]
        files=sorted(list(folder.glob('*.jpg'))+list(folder.glob('*.png')))
        if len(files)<FRAMES: raise RuntimeError(f'Thi?u frame: {folder} ({len(files)})')
        imgs=[]
        flip=self.train and random.random()<.5
        for p in files[:FRAMES]:
            im=cv2.imread(str(p),cv2.IMREAD_COLOR)
            if im is None: raise RuntimeError(f'Kh?ng ??c ???c {p}')
            im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB); im=cv2.resize(im,(IMG_DIM,IMG_DIM),interpolation=cv2.INTER_AREA)
            if flip: im=np.ascontiguousarray(im[:,::-1])
            t=torch.from_numpy(im).permute(2,0,1).float().div(255).sub(.5).div(.5); imgs.append(t)
        # [N,C,T,H,W], N=1; audio column is intentionally never opened
        return torch.stack(imgs,dim=1).unsqueeze(0), torch.tensor(label,dtype=torch.float32), video_id

train_ds=FGIVideoDataset(os.path.join(WORK_DIR,'train_split.csv'),True)
valid_ds=FGIVideoDataset(os.path.join(WORK_DIR,'valid_split.csv'))
test_ds=FGIVideoDataset(os.path.join(WORK_DIR,'test_imbalance_split.csv'))
print('Chunks:',len(train_ds),len(valid_ds),len(test_ds))
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
valid_loader=DataLoader(valid_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)

In [ ]:
model=FGIVideoOnly().to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-5)
criterion=nn.BCEWithLogitsLoss()
SCALER=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))
EPOCHS=30; BEST_PATH=os.path.join(OUT_DIR,'fgi_video_only_best.pth')

def epoch_run(loader,training=False):
    model.train(training); losses=[]; ys=[]; ps=[]; vids=[]
    for x,y,v in tqdm(loader,leave=False):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                logits=model(x); loss=criterion(logits,y)
            if training:
                optimizer.zero_grad(set_to_none=True); SCALER.scale(loss).backward(); SCALER.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),5); SCALER.step(optimizer); SCALER.update()
        losses.append(loss.item()*len(y)); ys.extend(y.detach().cpu().numpy()); ps.extend(torch.sigmoid(logits).detach().cpu().numpy()); vids.extend(v)
    auc=roc_auc_score(ys,ps) if len(set(ys))==2 else float('nan'); pred=(np.array(ps)>=.5)
    return {'loss':sum(losses)/len(loader.dataset),'auc':auc,'acc':accuracy_score(ys,pred),'f1':f1_score(ys,pred,zero_division=0),'y':ys,'p':ps,'vids':vids}

def video_metrics(r):
    d={}
    for y,p,v in zip(r['y'],r['p'],r['vids']): d.setdefault(v,{'y':y,'p':[]})['p'].append(p)
    yy=[x['y'] for x in d.values()]; pp=[np.mean(x['p']) for x in d.values()]; pred=np.array(pp)>=.5
    return {'videos':len(yy),'auc':roc_auc_score(yy,pp) if len(set(yy))==2 else float('nan'),'acc':accuracy_score(yy,pred),'f1':f1_score(yy,pred,zero_division=0)}

best=-1; history=[]
for ep in range(1,EPOCHS+1):
    tr=epoch_run(train_loader,True); va=epoch_run(valid_loader,False); vm=video_metrics(va)
    history.append({'epoch':ep,'train':{k:tr[k] for k in ('loss','auc','acc','f1')},'valid':{k:va[k] for k in ('loss','auc','acc','f1')},'valid_video':vm})
    print(f'Epoch {ep:02d} train={tr["auc"]:.4f} valid_chunk={va["auc"]:.4f} valid_video={vm["auc"]:.4f}')
    if vm['auc']>best:
        best=vm['auc']; torch.save({'model':model.state_dict(),'epoch':ep,'valid_video_auc':best,'config':{'crop':'mouth_lm','audio':False,'frames':FRAMES,'img_dim':IMG_DIM}},BEST_PATH); print('Saved:',BEST_PATH)

with open(os.path.join(OUT_DIR,'fgi_video_only_history.json'),'w') as f: json.dump(history,f)

In [ ]:
# ??nh gi? test chunk-level v? video-level
ckpt=torch.load(BEST_PATH,map_location=DEVICE); model.load_state_dict(ckpt['model'])
test=epoch_run(test_loader,False); print('Best epoch:',ckpt['epoch']); print('Test chunk:',{k:test[k] for k in ('loss','auc','acc','f1')}); print('Test video:',video_metrics(test))
print('Checkpoint:',BEST_PATH)

In [ ]:
from google.colab import files
files.download(BEST_PATH)